In [1]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnext101_32x8d, ResNeXt101_32X8D_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [2]:
DATA_DIR       = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split"
OUTPUT_DIR     = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101"
IMG_SIZE       = 224
BATCH_SIZE     = 8        # RTX 3050 4GB için güvenli değer
NUM_WORKERS    = 0         # Windows CMD için 0 olmalı         # CPU thread sayısı
PIN_MEMORY     = True
 
# Epoch sayıları (düşük tutuldu - hızlı test için)
FREEZE_EPOCHS  = 5         # Aşama 1: Backbone dondurulmuş
FINETUNE_EPOCHS = 5        # Aşama 2: Fine-tune
 
# Learning rate
FREEZE_LR      = 1e-3      # Aşama 1 LR
FINETUNE_LR    = 1e-5      # Aşama 2 LR (çok düşük!)
 
# Augmentation modu: "both" | "mixup" | "cutmix" | "none"
AUGMENTATION_MODE = "both"
 
# MixUp / CutMix parametreleri
MIXUP_ALPHA    = 0.4
CUTMIX_ALPHA   = 1.0
MIXUP_PROB     = 0.5       # "both" modunda MixUp seçilme olasılığı
 
SEED           = 42
SUBSET_RATIO   = 0.4       # Her epoch train setinin %40'ını kullan (hız için)

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
set_seed(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Kullanılan cihaz: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)


✅ Kullanılan cihaz: cuda
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   VRAM: 4.3 GB


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
 
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [6]:
# ─────────────────────────────────────────────
# SAFE DATASET - Hatalı dosyaları atlar
# ─────────────────────────────────────────────
from torchvision.datasets import ImageFolder
from torchvision.datasets.folder import default_loader
 
class SafeImageFolder(ImageFolder):
    """Açılamayan dosyaları sessizce atlayan ImageFolder"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Başlangıçta tüm dosyaları tara, açılamayanları çıkar
        print(f"   🔍 Dosyalar doğrulanıyor: {self.root}")
        valid_samples = []
        skipped = 0
        for path, label in tqdm(self.samples, desc="   Taranıyor", leave=False):
            try:
                with open(path, 'rb') as f:
                    f.read(10)  # Sadece açılabilir mi kontrol et
                valid_samples.append((path, label))
            except (FileNotFoundError, OSError):
                skipped += 1
        if skipped > 0:
            print(f"   ⚠️  {skipped} dosya atlandı (bulunamadı/açılamadı)")
        self.samples = valid_samples
        self.targets = [s[1] for s in valid_samples]
        print(f"   ✅ Geçerli dosya sayısı: {len(self.samples)}")

In [7]:
# ─────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────
print("\n📂 Veri seti yükleniyor...")
train_dataset = SafeImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = SafeImageFolder(os.path.join(DATA_DIR, "val"),   transform=val_test_transform)
test_dataset  = SafeImageFolder(os.path.join(DATA_DIR, "test"),  transform=val_test_transform)
 
NUM_CLASSES = len(train_dataset.classes)
print(f"   Sınıf sayısı : {NUM_CLASSES}")
print(f"   Train        : {len(train_dataset)} görsel")
print(f"   Validation   : {len(val_dataset)} görsel")
print(f"   Test         : {len(test_dataset)} görsel")


📂 Veri seti yükleniyor...
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\train


   ⚠️  1 dosya atlandı (bulunamadı/açılamadı)
   ✅ Geçerli dosya sayısı: 34019
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\val


   ✅ Geçerli dosya sayısı: 2043
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979
   Sınıf sayısı : 47
   Train        : 34019 görsel
   Validation   : 2043 görsel
   Test         : 1979 görsel


In [8]:
# ─────────────────────────────────────────────
# WEIGHTED RANDOM SAMPLER
# ─────────────────────────────────────────────
def get_weighted_sampler(dataset):
    targets = [s[1] for s in dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler
 
sampler = get_weighted_sampler(train_dataset)
 
# Subset: her epoch sadece SUBSET_RATIO kadar veri kullan
subset_size = int(len(train_dataset) * SUBSET_RATIO)
subset_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor([sampler.weights[i] for i in range(len(train_dataset))]),
    num_samples=subset_size,
    replacement=True
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=subset_sampler,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f"   Her epoch: {subset_size} görsel / {subset_size//BATCH_SIZE} batch (toplam {len(train_dataset)} görselin %{int(SUBSET_RATIO*100)}'i)")
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

   Her epoch: 13607 görsel / 1700 batch (toplam 34019 görselin %40'i)


In [9]:
# ─────────────────────────────────────────────
# MixUp / CutMix Fonksiyonları
# ─────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam
 
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    _, _, H, W = x.size()
    cut_rat = np.sqrt(1 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam
 
def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [10]:
weights = ResNeXt101_32X8D_Weights.DEFAULT
model = resnext101_32x8d(weights=weights)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnext101_32x8d-110c445d.pth" to C:\Users\ardac/.cache\torch\hub\checkpoints\resnext101_32x8d-110c445d.pth
100%|██████████| 340M/340M [00:47<00:00, 7.48MB/s] 


In [11]:
# ─────────────────────────────────────────────
# FREEZE / UNFREEZE Yardımcıları
# ─────────────────────────────────────────────
def freeze_backbone(model):
    """Sadece sınıflandırıcı katmanı eğitilebilir bırak"""
    for name, param in model.named_parameters():
        if 'fc' in name or 'head' in name or 'classifier' in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   ❄️  Backbone donduruldu. Eğitilebilir parametre: {trainable:,}")
 
def unfreeze_all(model):
    """Tüm parametreleri eğitilebilir yap"""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🔥 Tüm model açıldı. Eğitilebilir parametre: {trainable:,}")


In [12]:
# ─────────────────────────────────────────────
# METRİK HESAPLAMA
# ─────────────────────────────────────────────
def evaluate(model, loader, criterion, phase="val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
 
    return avg_loss, acc, prec, rec, f1, all_preds, all_labels

In [13]:
# ─────────────────────────────────────────────
# CONFUSION MATRIX & RAPOR KAYDET
# ─────────────────────────────────────────────
def save_confusion_matrix(labels, preds, class_names, save_path, title="Confusion Matrix"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, annot_kws={"size": 6})
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Tahmin", fontsize=10)
    ax.set_ylabel("Gerçek", fontsize=10)
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(rotation=0, fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   💾 Confusion matrix kaydedildi: {save_path}")
 
def save_metrics_plot(history, save_path, title="Eğitim Grafiği"):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14)
 
    metrics = [
        ('train_loss', 'val_loss', 'Loss'),
        ('train_acc',  'val_acc',  'Accuracy'),
        ('train_f1',   'val_f1',   'F1 Score'),
        ('train_prec', 'val_prec', 'Precision'),
        ('train_rec',  'val_rec',  'Recall'),
    ]
    for i, (tr_key, val_key, label) in enumerate(metrics):
        ax = axes[i // 3][i % 3]
        ax.plot(epochs, history[tr_key], 'b-o', label='Train')
        ax.plot(epochs, history[val_key], 'r-o', label='Val')
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True)
    axes[1][2].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"   💾 Grafik kaydedildi: {save_path}")


In [14]:
# ─────────────────────────────────────────────
# EĞİTİM FONKSİYONU
# ─────────────────────────────────────────────
def train_phase(model, train_loader, val_loader, optimizer, scheduler,
                criterion, num_epochs, phase_name, aug_mode):
 
    history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc',
                                 'train_f1','val_f1','train_prec','val_prec',
                                 'train_rec','val_rec']}
    best_val_f1 = 0.0
    best_model_path = os.path.join(OUTPUT_DIR, f"best_{phase_name}.pth")
 
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []
        start = time.time()
 
        skipped_batches = 0
        pbar = tqdm(train_loader, desc=f"[{phase_name}] Epoch {epoch}/{num_epochs}")
        for batch_data in pbar:
            try:
                images, labels = batch_data
                images, labels = images.to(device), labels.to(device)
 
                # Augmentation uygula
                use_aug = aug_mode != "none"
                if use_aug:
                    if aug_mode == "both":
                        use_mixup = random.random() < MIXUP_PROB
                    elif aug_mode == "mixup":
                        use_mixup = True
                    else:
                        use_mixup = False  # cutmix
 
                    if use_mixup:
                        images, y_a, y_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
                    else:
                        images, y_a, y_b, lam = cutmix_data(images, labels, CUTMIX_ALPHA)
 
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
                else:
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
 
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
 
                total_loss += loss.item()
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            except Exception:
                skipped_batches += 1
                continue
 
        if skipped_batches > 0:
            print(f"{skipped_batches} batch atlandı (dosya erişim hatası)")
 
        if scheduler:
            scheduler.step()
 
        # Train metrikleri
        tr_loss = total_loss / len(train_loader)
        tr_acc  = accuracy_score(all_labels, all_preds)
        tr_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
 
        # Val metrikleri
        val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader, criterion)
 
        elapsed = time.time() - start
        print(f"\n📊 [{phase_name}] Epoch {epoch}/{num_epochs} ({elapsed:.0f}s)")
        print(f"   Train → Loss:{tr_loss:.4f}  Acc:{tr_acc:.4f}  F1:{tr_f1:.4f}  Prec:{tr_prec:.4f}  Rec:{tr_rec:.4f}")
        print(f"   Val   → Loss:{val_loss:.4f}  Acc:{val_acc:.4f}  F1:{val_f1:.4f}  Prec:{val_prec:.4f}  Rec:{val_rec:.4f}")
 
        # History kaydet
        history['train_loss'].append(tr_loss);  history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc);    history['val_acc'].append(val_acc)
        history['train_f1'].append(tr_f1);      history['val_f1'].append(val_f1)
        history['train_prec'].append(tr_prec);  history['val_prec'].append(val_prec)
        history['train_rec'].append(tr_rec);    history['val_rec'].append(val_rec)
 
        # En iyi modeli kaydet
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"   ✅ En iyi model kaydedildi (Val F1: {best_val_f1:.4f})")
 
    return history, best_model_path

In [15]:
# ─────────────────────────────────────────────
# TEST DEĞERLENDİRME
# ─────────────────────────────────────────────
def test_evaluation(model, test_loader, criterion, class_names, phase_label, aug_label):
    print(f"\n{'='*60}")
    print(f"🧪 TEST DEĞERLENDİRMESİ - {phase_label} | Aug: {aug_label}")
    print(f"{'='*60}")
    _, acc, prec, rec, f1, preds, labels = evaluate(model, test_loader, criterion, "test")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    print(f"   F1 Score  : {f1:.4f}")
 
    # Confusion matrix
    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{phase_label}_{aug_label}.png")
    save_confusion_matrix(labels, preds, class_names, cm_path,
                          title=f"Confusion Matrix - {phase_label} | {aug_label}")
 
    # Classification report
    report = classification_report(labels, preds, target_names=class_names, zero_division=0)
    report_path = os.path.join(OUTPUT_DIR, f"classification_report_{phase_label}_{aug_label}.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"Phase: {phase_label} | Augmentation: {aug_label}\n\n")
        f.write(report)
    print(f"   💾 Classification report: {report_path}")
 
    return acc, prec, rec, f1


In [17]:
# ─────────────────────────────────────────────
# ANA AKIŞ
# ─────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
class_names = train_dataset.classes
 
print(f"\n{'='*60}")
print(f"🚀 EĞİTİM BAŞLIYOR")
print(f"   Model          : ResNext101e")
print(f"   Augmentation   : {AUGMENTATION_MODE}")
print(f"   Freeze Epochs  : {FREEZE_EPOCHS}")
print(f"   Finetune Epochs: {FINETUNE_EPOCHS}")
print(f"   Batch Size     : {BATCH_SIZE}")
print(f"{'='*60}\n")


🚀 EĞİTİM BAŞLIYOR
   Model          : ResNext101e
   Augmentation   : both
   Freeze Epochs  : 5
   Finetune Epochs: 5
   Batch Size     : 8



In [18]:
# ──────────────────────────────
# AŞAMA 1: BACKBONE FREEZE
# ──────────────────────────────
print("\n" + "="*60)
print("⚡ AŞAMA 1: BACKBONE FREEZE")
print("="*60)
freeze_backbone(model)
 
optimizer_freeze = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FREEZE_LR, weight_decay=1e-4
)
scheduler_freeze = optim.lr_scheduler.CosineAnnealingLR(optimizer_freeze, T_max=FREEZE_EPOCHS)
 
history_freeze, best_freeze_path = train_phase(
    model, train_loader, val_loader,
    optimizer_freeze, scheduler_freeze,
    criterion, FREEZE_EPOCHS,
    phase_name="freeze",
    aug_mode=AUGMENTATION_MODE
)
 
# Grafik kaydet
save_metrics_plot(history_freeze,
                  os.path.join(OUTPUT_DIR, "grafik_freeze.png"),
                  title=f"Aşama 1: Backbone Freeze | Aug: {AUGMENTATION_MODE}")
 
# Aşama 1 test değerlendirmesi
print("\n📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_freeze_path))
acc1, prec1, rec1, f1_1 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase1_Freeze", aug_label=AUGMENTATION_MODE
)



⚡ AŞAMA 1: BACKBONE FREEZE
   ❄️  Backbone donduruldu. Eğitilebilir parametre: 96,303


[freeze] Epoch 1/5: 100%|██████████| 1701/1701 [04:44<00:00,  5.98it/s, loss=3.0039]



📊 [freeze] Epoch 1/5 (319s)
   Train → Loss:2.7210  Acc:0.2635  F1:0.2556  Prec:0.2563  Rec:0.2620
   Val   → Loss:1.6719  Acc:0.5888  F1:0.5132  Prec:0.5931  Rec:0.5352
   ✅ En iyi model kaydedildi (Val F1: 0.5132)


[freeze] Epoch 2/5: 100%|██████████| 1701/1701 [05:40<00:00,  4.99it/s, loss=1.3123]



📊 [freeze] Epoch 2/5 (377s)
   Train → Loss:2.2342  Acc:0.3585  F1:0.3518  Prec:0.3495  Rec:0.3579
   Val   → Loss:1.4949  Acc:0.6143  F1:0.5385  Prec:0.5849  Rec:0.5595
   ✅ En iyi model kaydedildi (Val F1: 0.5385)


...


📊 [freeze] Epoch 4/5 (337s)
   Train → Loss:2.0185  Acc:0.3983  F1:0.3933  Prec:0.3923  Rec:0.3970
   Val   → Loss:1.3550  Acc:0.6525  F1:0.5870  Prec:0.6221  Rec:0.6002


[freeze] Epoch 5/5: 100%|██████████| 1701/1701 [04:57<00:00,  5.73it/s, loss=2.4238]



📊 [freeze] Epoch 5/5 (332s)
   Train → Loss:2.0092  Acc:0.4028  F1:0.4002  Prec:0.3989  Rec:0.4042
   Val   → Loss:1.3988  Acc:0.6569  F1:0.5905  Prec:0.6174  Rec:0.6029
   💾 Grafik kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\grafik_freeze.png

📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...


C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\606711733.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_freeze_path))



🧪 TEST DEĞERLENDİRMESİ - Phase1_Freeze | Aug: both
   Accuracy  : 0.6534
   Precision : 0.6187
   Recall    : 0.6010
   F1 Score  : 0.5882
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\confusion_matrix_Phase1_Freeze_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\classification_report_Phase1_Freeze_both.txt


In [19]:
# ──────────────────────────────
# AŞAMA 2: FINE-TUNING
# ──────────────────────────────
print("\n" + "="*60)
print("🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)")
print("="*60)
unfreeze_all(model)
 
optimizer_finetune = optim.AdamW(
    model.parameters(),
    lr=FINETUNE_LR, weight_decay=1e-4
)
scheduler_finetune = optim.lr_scheduler.CosineAnnealingLR(optimizer_finetune, T_max=FINETUNE_EPOCHS)
 
history_finetune, best_finetune_path = train_phase(
    model, train_loader, val_loader,
    optimizer_finetune, scheduler_finetune,
    criterion, FINETUNE_EPOCHS,
    phase_name="finetune",
    aug_mode=AUGMENTATION_MODE
)
 
# Grafik kaydet
save_metrics_plot(history_finetune,
                  os.path.join(OUTPUT_DIR, "grafik_finetune.png"),
                  title=f"Aşama 2: Fine-Tuning | Aug: {AUGMENTATION_MODE}")
 
# Aşama 2 test değerlendirmesi
print("\n📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_finetune_path))
acc2, prec2, rec2, f1_2 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase2_FineTune", aug_label=AUGMENTATION_MODE
)


🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)
   🔥 Tüm model açıldı. Eğitilebilir parametre: 86,838,639


[finetune] Epoch 1/5: 100%|██████████| 1701/1701 [1:09:15<00:00,  2.44s/it, loss=2.9551]



📊 [finetune] Epoch 1/5 (4193s)
   Train → Loss:1.8911  Acc:0.4316  F1:0.4280  Prec:0.4295  Rec:0.4317
   Val   → Loss:1.0208  Acc:0.7293  F1:0.6816  Prec:0.6991  Rec:0.6887
   ✅ En iyi model kaydedildi (Val F1: 0.6816)


[finetune] Epoch 2/5: 100%|██████████| 1701/1701 [14:32<00:00,  1.95it/s, loss=0.7176]



📊 [finetune] Epoch 2/5 (910s)
   Train → Loss:1.7356  Acc:0.4756  F1:0.4733  Prec:0.4730  Rec:0.4754
   Val   → Loss:0.9142  Acc:0.7455  F1:0.7005  Prec:0.7090  Rec:0.7082
   ✅ En iyi model kaydedildi (Val F1: 0.7005)


...


📊 [finetune] Epoch 4/5 (896s)
   Train → Loss:1.6046  Acc:0.5211  F1:0.5184  Prec:0.5181  Rec:0.5204
   Val   → Loss:0.8343  Acc:0.7675  F1:0.7229  Prec:0.7297  Rec:0.7273
   ✅ En iyi model kaydedildi (Val F1: 0.7229)


[finetune] Epoch 5/5: 100%|██████████| 1701/1701 [12:58<00:00,  2.18it/s, loss=0.6845]



📊 [finetune] Epoch 5/5 (813s)
   Train → Loss:1.5892  Acc:0.5161  F1:0.5142  Prec:0.5137  Rec:0.5163
   Val   → Loss:0.8168  Acc:0.7709  F1:0.7263  Prec:0.7348  Rec:0.7324
   ✅ En iyi model kaydedildi (Val F1: 0.7263)
   💾 Grafik kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\grafik_finetune.png

📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...


C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\264694363.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_finetune_path))



🧪 TEST DEĞERLENDİRMESİ - Phase2_FineTune | Aug: both
   Accuracy  : 0.7660
   Precision : 0.7351
   Recall    : 0.7304
   F1 Score  : 0.7238
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\confusion_matrix_Phase2_FineTune_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\classification_report_Phase2_FineTune_both.txt


In [20]:
# ──────────────────────────────
# KARŞILAŞTIRMA TABLOSU
# ──────────────────────────────
print("\n" + "="*60)
print("📈 SONUÇ KARŞILAŞTIRMASI")
print("="*60)
print(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-"*60)
print(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}")
print(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}")
print("="*60)
 
# Özet dosyası kaydet
summary_path = os.path.join(OUTPUT_DIR, f"summary_{AUGMENTATION_MODE}.txt")
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"ResNeSt101 Eğitim Özeti\n")
    f.write(f"Augmentation: {AUGMENTATION_MODE}\n")
    f.write(f"{'='*60}\n")
    f.write(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}\n")
    f.write(f"{'-'*60}\n")
    f.write(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}\n")
    f.write(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}\n")
 
print(f"\n✅ Tüm sonuçlar kaydedildi: {OUTPUT_DIR}")
print(f"   Özet dosyası: {summary_path}")
print("\n🎉 EĞİTİM TAMAMLANDI!")


📈 SONUÇ KARŞILAŞTIRMASI
Aşama                               Acc     Prec      Rec       F1
------------------------------------------------------------
Phase 1 (Freeze)                 0.6534   0.6187   0.6010   0.5882
Phase 2 (Fine-Tune)              0.7660   0.7351   0.7304   0.7238

✅ Tüm sonuçlar kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101
   Özet dosyası: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\summary_both.txt

🎉 EĞİTİM TAMAMLANDI!


In [21]:
# ─────────────────────────────────────────────
# GRAD-CAM DEĞERLENDİRMESİ
# ─────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import cv2
 
GRADCAM_DIR = os.path.join(OUTPUT_DIR, "gradcam")
os.makedirs(GRADCAM_DIR, exist_ok=True)
 
SAMPLES_PER_CLASS = 2   # Her sınıftan kaç görsel
 
# Grad-CAM için normalize edilmemiş transform (görsel gösterimi için)
vis_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
 
norm_transform = transforms.Normalize(
    [0.485, 0.456, 0.406],
    [0.229, 0.224, 0.225]
)
 
def get_target_layer(model):
    """ResNeSt101 için son conv katmanını döndür"""
    return [model.layer4[-1]]
 
def run_gradcam(model, model_name, dataset, class_names):
    """Verilen model için her sınıftan SAMPLES_PER_CLASS görsel üzerinde Grad-CAM uygula"""
    model.eval()
    target_layers = get_target_layer(model)
    cam = GradCAM(model=model, target_layers=target_layers)
 
    save_dir = os.path.join(GRADCAM_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)
 
    # Her sınıftan örnek topla
    class_samples = {i: [] for i in range(len(class_names))}
    for path, label in dataset.samples:
        if len(class_samples[label]) < SAMPLES_PER_CLASS:
            class_samples[label].append(path)
        if all(len(v) >= SAMPLES_PER_CLASS for v in class_samples.values()):
            break
 
    print(f"\n🎨 [{model_name}] Grad-CAM oluşturuluyor...")
    for class_idx, paths in tqdm(class_samples.items(), desc=model_name):
        class_name = class_names[class_idx]
        fig, axes = plt.subplots(len(paths), 3, figsize=(12, 4 * len(paths)))
        if len(paths) == 1:
            axes = [axes]
        fig.suptitle(f"{model_name} | Sınıf: {class_name}", fontsize=11)
 
        for row, path in enumerate(paths):
            try:
                # Orijinal görseli yükle
                from PIL import Image as PILImage
                pil_img = PILImage.open(path).convert("RGB")
                pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
                rgb_img = np.array(pil_img).astype(np.float32) / 255.0
 
                # Model için tensor
                input_tensor = norm_transform(vis_transform(pil_img)).unsqueeze(0).to(device)
 
                # Tahmin
                with torch.no_grad():
                    output = model(input_tensor)
                pred_idx = output.argmax(dim=1).item()
                pred_name = class_names[pred_idx]
                confidence = torch.softmax(output, dim=1)[0][pred_idx].item()
 
                # Grad-CAM
                targets = [ClassifierOutputTarget(class_idx)]
                grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
                grayscale_cam = grayscale_cam[0]
                cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
 
                # Doğru mu yanlış mı
                is_correct = (pred_idx == class_idx)
                border_color = "green" if is_correct else "red"
                status = "✅ Doğru" if is_correct else f"❌ Yanlış → {pred_name}"
 
                # Görselleri çiz
                axes[row][0].imshow(pil_img)
                axes[row][0].set_title("Orijinal", fontsize=9)
                axes[row][0].axis("off")
 
                axes[row][1].imshow(grayscale_cam, cmap="jet")
                axes[row][1].set_title("Grad-CAM Isı Haritası", fontsize=9)
                axes[row][1].axis("off")
 
                axes[row][2].imshow(cam_image)
                axes[row][2].set_title(
                    f"{status}\nGüven: {confidence:.2%}", fontsize=9,
                    color=border_color
                )
                axes[row][2].axis("off")
                for spine in axes[row][2].spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(3)
 
            except Exception as e:
                print(f"   ⚠️  {path} atlandı: {e}")
                continue
 
        plt.tight_layout()
        safe_name = class_name.replace("/", "_").replace("\\", "_")
        save_path = os.path.join(save_dir, f"{safe_name}.png")
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        plt.close()
 
    print(f"   💾 Kaydedildi: {save_dir}")

In [22]:
# ─────────────────────────────────────────────
# MODELLERİ YÜKLE VE GRAD-CAM ÇALIŞTIR
# ─────────────────────────────────────────────
FREEZE_PATH   = os.path.join(OUTPUT_DIR, "best_freeze.pth")
FINETUNE_PATH = os.path.join(OUTPUT_DIR, "best_finetune.pth")
 
# Test dataseti kullan (augmentation yok, temiz görseller)
gradcam_dataset = SafeImageFolder(
    os.path.join(DATA_DIR, "test"),
    transform=val_test_transform
)
 
print("\n" + "="*60)
print("🔍 GRAD-CAM BAŞLIYOR")
print("="*60)
print(f"   Her sınıftan {SAMPLES_PER_CLASS} görsel x {len(train_dataset.classes)} sınıf")
print(f"   Toplam: ~{SAMPLES_PER_CLASS * len(train_dataset.classes)} görsel x 2 model")
 
# Backbone Freeze modeli
print("\n⚡ Backbone Freeze modeli yükleniyor...")
model.load_state_dict(torch.load(FREEZE_PATH, map_location=device))
run_gradcam(model, "Phase1_BackboneFreeze", gradcam_dataset, train_dataset.classes)
 
# Fine-Tune modeli
print("\n🔥 Fine-Tune modeli yükleniyor...")
model.load_state_dict(torch.load(FINETUNE_PATH, map_location=device))
run_gradcam(model, "Phase2_FineTune", gradcam_dataset, train_dataset.classes)
 
print("\n" + "="*60)
print(f"✅ GRAD-CAM TAMAMLANDI!")
print(f"   Sonuçlar: {GRADCAM_DIR}")
print(f"   Phase1_BackboneFreeze/ → Freeze modeli görselleri")
print(f"   Phase2_FineTune/       → Fine-tune modeli görselleri")
print("="*60)

   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\257219615.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(FREEZE_PATH, map_location=de

   ✅ Geçerli dosya sayısı: 1979

🔍 GRAD-CAM BAŞLIYOR
   Her sınıftan 2 görsel x 47 sınıf
   Toplam: ~94 görsel x 2 model

⚡ Backbone Freeze modeli yükleniyor...

🎨 [Phase1_BackboneFreeze] Grad-CAM oluşturuluyor...


Phase1_BackboneFreeze:   0%|          | 0/47 [00:00<?, ?it/s]C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\241420590.py:106: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\241420590.py:106: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\241420590.py:109: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=100, bbox_inches="tight")
C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\241420590.py:109: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=100, bbox_inches="tight")
Phase1_BackboneFreeze: 100%|██████████| 47/47 [01:33<00:00,  1.99s/it]
C:\Users\ardac\AppData\Local\Temp\ipykernel_9776\257219615.py:26: FutureWarning: You are using `torch.load` with `weights_

   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\gradcam\Phase1_BackboneFreeze

🔥 Fine-Tune modeli yükleniyor...

🎨 [Phase2_FineTune] Grad-CAM oluşturuluyor...


Phase2_FineTune: 100%|██████████| 47/47 [00:52<00:00,  1.11s/it]

   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\gradcam\Phase2_FineTune

✅ GRAD-CAM TAMAMLANDI!
   Sonuçlar: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_resnext101\gradcam
   Phase1_BackboneFreeze/ → Freeze modeli görselleri
   Phase2_FineTune/       → Fine-tune modeli görselleri
